## Preliminaries
### Import statements

In [42]:
from osfclient import OSF
import pandas as pd
import os

### Global values

In [43]:
OSF_PROJECT = "uz6hg"
TEMP_DIR = "temp"
OSF_FILES = ["tokens.tsv", "accent_corrections.tsv", "lemma_corrections.tsv"]

### Check for temporary directory; create if necessary

To avoid overwriting output from other notebooks, we're saving to a temporary directory.

In [44]:
if not os.path.exists(TEMP_DIR):
    os.mkdir(TEMP_DIR)

## Load token data
### Download data files from OSF

In [45]:
osf = OSF()

project = osf.project(OSF_PROJECT)
storage = project.storage()
for file in storage.files:
    if file.name in OSF_FILES:
        with open(os.path.join(TEMP_DIR, file.name), "wb") as fh:
            file.write_to(fh)

100%|██████████████████████████████████| 73.1M/73.1M [00:04<00:00, 15.3Mbytes/s]
100%|██████████████████████████████████| 9.50k/9.50k [00:00<00:00, 11.4Mbytes/s]
100%|██████████████████████████████████| 39.9k/39.9k [00:00<00:00, 20.4Mbytes/s]


### Load the token table

In [46]:
tokens = pd.read_csv(os.path.join(TEMP_DIR, "tokens.tsv"), sep="\t", dtype=str)

### Corrections

In [47]:
# remove punctuation
tokens = tokens.loc[~(tokens["pos"]=="PUNCT")]

# correct elisions and accents
for corr_file in ["accent_corrections.tsv", "lemma_corrections.tsv"]:
    with open(os.path.join(TEMP_DIR, corr_file)) as f:
        for row in f.readlines():
            if not row.startswith("#"):
                old, new = row.strip().split("\t")
                tokens.loc[tokens["lemma"]==old, "lemma"] = new

# force Triphiodorus pref to string
tokens.loc[tokens["work"]=="Sack of Troy", "pref"] = " "

### Lemma count

In [48]:
corpus_lemma_count = tokens["lemma"].value_counts()

In [49]:
corpus_lemma_count

lemma
δέ              21339
καί             13671
ὁ                9589
τε               5574
ἐγώ              3783
                ...  
μιἥψκω              1
νεῖσθʼ              1
παρέννεπεν          1
υἷές                1
ἀρειοτέροισι        1
Name: count, Length: 36646, dtype: int64

## Lemma analysis
### Hapax legomena per author

In [50]:
authors = ["Homer", "Apollonius", "Triphiodorus", "Quintus", "Nonnus"]
hapax_legomena = corpus_lemma_count[corpus_lemma_count==1].index.values
df = pd.DataFrame(dict(
    hapax = tokens.loc[tokens.lemma.isin(hapax_legomena), "author"].value_counts()[authors],
    tokens = tokens["author"].value_counts()[authors],
    ))
df["freq"] = df["hapax"].div(df["tokens"]) * 1000
display(df)

,hapax,tokens,freq
author,,,
Homer,6729,199029,33.809143
Apollonius,3092,38820,79.649665
Triphiodorus,373,4232,88.137996
Quintus,2850,60116,47.408344
Nonnus,7043,127577,55.205876


### Author-specific lemmata 
#### Including hapax legomena

In [52]:
lemma_count_by_author = pd.crosstab(tokens["lemma"], tokens["work"]).loc[corpus_lemma_count.index.values]

single_auth_lemmas = (lemma_count_by_author
    .gt(0) # true if author uses lemma, false otherwise
    .sum(axis=1) # count how many authors use each lemma
    .loc[lambda s: s==1] # filter for lemmas with single author
    .index.values # keep just the list of lemmas
)

df = (lemma_count_by_author
        .loc[single_auth_lemmas] # select only single-author lemmas
        .assign(_key=lambda d: d.sum(axis=1)) # temp column for sorting by count in any author
        .sort_values("_key", ascending=False) # sort
        .drop(columns="_key") # get rid of temp column after sorting
)

df.to_csv(os.path.join(TEMP_DIR, "author-specific-lemmata.tsv"), sep="\t")
display(df)

work,Argonautica,Dionysiaca,Iliad,Odyssey,Posthomerica,Sack of Troy
lemma,,,,,,
Βάκχος,0,366,0,0,0,0
Λυαῖος,0,212,0,0,0,0
Τηλέμαχος,0,0,0,177,0,0
Ἰνδῶ,0,135,0,0,0,0
θύρσος,0,132,0,0,0,0
...,...,...,...,...,...,...
δεδὰ,0,1,0,0,0,0
αὐχενί,0,1,0,0,0,0
εὐρυβάτοιο,0,1,0,0,0,0


### Lexical overlap

What fraction of each author's lexicon (rows) is used by each of the others (cols)? 

In [53]:
bool_df = lemma_count_by_author.gt(0).astype(int)
(bool_df.T @ bool_df).div(bool_df.sum())

work,Argonautica,Dionysiaca,Iliad,Odyssey,Posthomerica,Sack of Troy
work,,,,,,
Argonautica,1.000000,0.181648,0.303430,0.294424,0.324837,0.495276
Dionysiaca,0.365882,1.000000,0.334511,0.300241,0.354950,0.643958
Iliad,0.343412,0.187956,1.000000,0.388187,0.383758,0.562407
Odyssey,0.345412,0.174873,0.402391,1.000000,0.347836,0.526106
Posthomerica,0.322353,0.174873,0.336486,0.294224,1.000000,0.530582
Sack of Troy,0.117176,0.075638,0.117568,0.106097,0.126497,1.000000


#### How big an effect do rare words have on overlap?

In [54]:
# minimum number of occurrences corpus-wide
cutoff = 5

bool_df = (lemma_count_by_author
    .loc[lemma_count_by_author.index.isin(corpus_lemma_count[corpus_lemma_count>cutoff].index.values)]
    .gt(0)
    .astype(int)
)

(bool_df.T @ bool_df).div(bool_df.sum())

work,Argonautica,Dionysiaca,Iliad,Odyssey,Posthomerica,Sack of Troy
work,,,,,,
Argonautica,1.000000,0.512041,0.642161,0.663946,0.686747,0.716966
Dionysiaca,0.756791,1.000000,0.698551,0.696594,0.725000,0.874120
Iliad,0.760849,0.559992,1.000000,0.838728,0.800904,0.826427
Odyssey,0.736497,0.522814,0.785244,1.000000,0.741566,0.776388
Posthomerica,0.711833,0.508450,0.700659,0.692936,1.000000,0.771697
Sack of Troy,0.286294,0.236164,0.278524,0.279482,0.297289,1.000000


#### What words are used by all authors?

In [55]:
lemma_count_by_author.loc[lemma_count_by_author.gt(0).sum(axis=1)==6].to_clipboard()